# Analisis Tren Harga Komoditas Pangan di Kawasan ASEAN Berbasis Data Warehouse melalui Integrasi Data WFP dan World Bank dengan Pendekatan Star Schema
**UAS Data Warehouse 2025/2026 | S1 Sains Data UNESA**

| Fase | Keterangan |
|------|------------|
| 1    | Extract — WFP CSV + World Bank API |
| 2    | Transform — cleaning, integrasi, star schema |
| 3    | Load — feeder ke PostgreSQL/Supabase |

## 0. Setup

In [10]:
import sys
import os
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

print('DATABASE_URL tersedia:', bool(os.getenv('DATABASE_URL')))

DATABASE_URL tersedia: True


## 1. Extract

### 1a. WFP Dataset

Link: https://www.kaggle.com/datasets/abhishekgupta56447/global-food-prices-database-wfp

In [11]:
import subprocess

batches_wfp = [
    ('1', '2024', '2024'),
    ('2', '2025', '2025'),
    ('3', '2024', '2025'),
]

for batch, start, end in batches_wfp:
    result = subprocess.run(
        ['python', 'scraper/reader_wfp.py', '--batch', batch, '--start', start, '--end', end],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)

[reader_wfp] Membaca file: data\raw\wfp_food_prices_global_2024.csv
[reader_wfp] Total baris dimuat: 437,038
[reader_wfp] Baris setelah filter (2024–2024, ASEAN): 81,677
[reader_wfp] Tersimpan -> data\raw\raw_wfp_batch_1.csv

[reader_wfp] Membaca file: data\raw\wfp_food_prices_global_2025.csv
[reader_wfp] Total baris dimuat: 419,527
[reader_wfp] Baris setelah filter (2025–2025, ASEAN): 69,495
[reader_wfp] Tersimpan -> data\raw\raw_wfp_batch_2.csv

[reader_wfp] Membaca file: data\raw\wfp_food_prices_global_2024.csv
[reader_wfp] Total baris dimuat: 437,038
[reader_wfp] Membaca file: data\raw\wfp_food_prices_global_2025.csv
[reader_wfp] Total baris dimuat: 419,527
[reader_wfp] Baris setelah filter (2024–2025, ASEAN): 151,172
[reader_wfp] Tersimpan -> data\raw\raw_wfp_batch_3.csv



### 1b. World Bank API

In [12]:
batches_wb = [
    ('1', '2024', '2024', 'FP.CPI.TOTL.ZG'),
    ('2', '2025', '2025', 'NY.GDP.PCAP.CD'),
    ('3', '2024', '2025', 'all'),
]

for batch, start, end, indicator in batches_wb:
    result = subprocess.run(
        ['python', 'scraper/reader_worldbank.py',
         '--batch', batch, '--start', start, '--end', end, '--indicator', indicator],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)

[reader_worldbank] GET FP.CPI.TOTL.ZG | tahun 2024–2024 | halaman 1
[reader_worldbank] Total record diterima untuk FP.CPI.TOTL.ZG: 10
[reader_worldbank] Tersimpan -> data\raw\raw_worldbank_batch_1_FP_CPI_TOTL_ZG.json

[reader_worldbank] GET NY.GDP.PCAP.CD | tahun 2025–2025 | halaman 1
[reader_worldbank] Total record diterima untuk NY.GDP.PCAP.CD: 10
[reader_worldbank] Tersimpan -> data\raw\raw_worldbank_batch_2_NY_GDP_PCAP_CD.json

[reader_worldbank] GET FP.CPI.TOTL.ZG | tahun 2024–2025 | halaman 1
[reader_worldbank] Total record diterima untuk FP.CPI.TOTL.ZG: 20
[reader_worldbank] Tersimpan -> data\raw\raw_worldbank_batch_3_FP_CPI_TOTL_ZG.json
[reader_worldbank] GET NY.GDP.PCAP.CD | tahun 2024–2025 | halaman 1
[reader_worldbank] Total record diterima untuk NY.GDP.PCAP.CD: 20
[reader_worldbank] Tersimpan -> data\raw\raw_worldbank_batch_3_NY_GDP_PCAP_CD.json
[reader_worldbank] GET NY.GDP.MKTP.KD.ZG | tahun 2024–2025 | halaman 1
[reader_worldbank] Total record diterima untuk NY.GDP.MKTP.

### 1c. Verifikasi file raw

In [13]:
import glob
import os

raw_files = sorted(glob.glob('data/raw/*'))
print(f'Total file di data/raw/: {len(raw_files)}')
for f in raw_files:
    size_kb = os.path.getsize(f) / 1024
    print(f'  {os.path.basename(f):50s} {size_kb:8.1f} KB')

Total file di data/raw/: 11
  raw_wfp_batch_1.csv                                 11639.0 KB
  raw_wfp_batch_2.csv                                  9861.4 KB
  raw_wfp_batch_3.csv                                 21500.2 KB
  raw_worldbank_batch_1_FP_CPI_TOTL_ZG.json               3.3 KB
  raw_worldbank_batch_2_NY_GDP_PCAP_CD.json               3.1 KB
  raw_worldbank_batch_3_FP_CPI_TOTL_ZG.json               6.5 KB
  raw_worldbank_batch_3_NY_GDP_MKTP_KD_ZG.json            6.3 KB
  raw_worldbank_batch_3_NY_GDP_PCAP_CD.json               6.4 KB
  raw_worldbank_batch_3_TM_VAL_FOOD_ZS_UN.json            6.6 KB
  wfp_food_prices_global_2024.csv                     56771.2 KB
  wfp_food_prices_global_2025.csv                     54487.1 KB


## 2. Transform

In [14]:
result = subprocess.run(
    ['python', 'etl/preprocess.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

[preprocess] Memulai Fase 2 — Transform
[preprocess] Memuat WFP: data\raw\raw_wfp_batch_1.csv
[preprocess] Memuat WFP: data\raw\raw_wfp_batch_2.csv
[preprocess] Memuat WFP: data\raw\raw_wfp_batch_3.csv
[preprocess] Total baris WFP sebelum cleaning: 302,344
[preprocess] Baris WFP setelah cleaning: 151,172
[preprocess] Memuat World Bank: data\raw\raw_worldbank_batch_1_FP_CPI_TOTL_ZG.json
[preprocess] Memuat World Bank: data\raw\raw_worldbank_batch_2_NY_GDP_PCAP_CD.json
[preprocess] Memuat World Bank: data\raw\raw_worldbank_batch_3_FP_CPI_TOTL_ZG.json
[preprocess] Memuat World Bank: data\raw\raw_worldbank_batch_3_NY_GDP_MKTP_KD_ZG.json
[preprocess] Memuat World Bank: data\raw\raw_worldbank_batch_3_NY_GDP_PCAP_CD.json
[preprocess] Memuat World Bank: data\raw\raw_worldbank_batch_3_TM_VAL_FOOD_ZS_UN.json
[preprocess] Total baris World Bank setelah load: 36

[preprocess] Membangun tabel dimensi...

[preprocess] Membangun tabel fakta...
[preprocess] Baris fact_harga_pangan: 3,834

[preprocess]

### Verifikasi output processed

In [15]:
import pandas as pd

tables = ['fact_harga_pangan', 'dim_waktu', 'dim_negara', 'dim_komoditas', 'dim_indikator']

for t in tables:
    path = f'data/processed/{t}.csv'
    try:
        df = pd.read_csv(path)
        print(f'{t:30s}: {len(df):,} baris | kolom: {list(df.columns)}')
    except FileNotFoundError:
        print(f'{t:30s}: FILE TIDAK DITEMUKAN')

fact_harga_pangan             : 3,834 baris | kolom: ['waktu_id', 'negara_id', 'komoditas_id', 'avg_price', 'min_price', 'max_price', 'record_count', 'fp_cpi_totl_zg', 'ny_gdp_mktp_kd_zg', 'ny_gdp_pcap_cd', 'tm_val_food_zs_un']
dim_waktu                     : 24 baris | kolom: ['waktu_id', 'year', 'month', 'quarter', 'periode']
dim_negara                    : 6 baris | kolom: ['negara_id', 'country_name', 'region']
dim_komoditas                 : 152 baris | kolom: ['komoditas_id', 'commodity_name', 'category']
dim_indikator                 : 4 baris | kolom: ['indikator_id', 'indicator_code', 'indicator_name']


In [16]:
# Preview fact table
fact = pd.read_csv('data/processed/fact_harga_pangan.csv')
print(f'Shape: {fact.shape}')
display(fact.head(10))
display(fact.describe())

Shape: (3834, 11)


,waktu_id,negara_id,komoditas_id,avg_price,min_price,max_price,record_count,fp_cpi_totl_zg,ny_gdp_mktp_kd_zg,ny_gdp_pcap_cd,tm_val_food_zs_un
0,1,1,25,61209.4116,30277.78,198888.89,207,NaN,NaN,NaN,NaN
1,1,1,26,47910.4270,16666.67,137500.00,190,NaN,NaN,NaN,NaN
2,1,1,27,73850.0086,38333.33,300000.00,178,NaN,NaN,NaN,NaN
3,1,1,28,58902.0840,26000.00,140000.00,207,NaN,NaN,NaN,NaN
4,1,1,29,62055.7539,32500.00,110000.00,161,NaN,NaN,NaN,NaN
5,1,1,35,28267.2729,21200.00,42500.00,207,NaN,NaN,NaN,NaN
6,1,1,36,28267.9630,21200.00,42500.00,207,NaN,NaN,NaN,NaN
7,1,1,57,40294.1933,22450.00,60000.00,207,NaN,NaN,NaN,NaN
8,1,1,58,40293.7619,22450.00,60000.00,207,NaN,NaN,NaN,NaN
9,1,1,73,132564.9801,85000.00,165000.00,191,NaN,NaN,NaN,NaN


,waktu_id,negara_id,komoditas_id,avg_price,min_price,max_price,record_count,fp_cpi_totl_zg,ny_gdp_mktp_kd_zg,ny_gdp_pcap_cd,tm_val_food_zs_un
count,3834.000000,3834.000000,3834.000000,3834.000000,3834.000000,3834.000000,3834.000000,0.0,0.0,0.0,0.0
mean,12.241002,3.703704,76.940793,11664.125015,8831.779919,15572.616437,39.429317,NaN,NaN,NaN,NaN
std,7.078688,1.609829,43.870828,25311.164420,19874.774263,32445.697582,43.694261,NaN,NaN,NaN,NaN
min,1.000000,1.000000,1.000000,0.177300,0.140000,0.200000,1.000000,NaN,NaN,NaN,NaN
25%,6.000000,2.000000,40.000000,77.846725,43.812500,122.250000,13.000000,NaN,NaN,NaN,NaN
50%,12.000000,4.000000,76.000000,1621.841800,700.000000,2500.000000,24.000000,NaN,NaN,NaN,NaN
75%,18.000000,5.000000,115.000000,8604.652425,5831.000000,13132.500000,52.000000,NaN,NaN,NaN,NaN
max,24.000000,6.000000,152.000000,143772.613100,115000.000000,300000.000000,218.000000,NaN,NaN,NaN,NaN


## 3. Load

In [21]:
result = subprocess.run(
    ['python', 'etl/feeder.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

[feeder] Koneksi ke PostgreSQL berhasil.
[feeder] DDL berhasil dijalankan dari sql/ddl.sql

[feeder] Memulai load data...
[feeder] dim_waktu: 24 baris dimuat.
[feeder] dim_negara: 6 baris dimuat.
[feeder] dim_komoditas: 152 baris dimuat.
[feeder] dim_indikator: 4 baris dimuat.
[feeder] fact_harga_pangan: 3,834 baris dimuat.

[feeder] Fase 3 selesai. Koneksi ditutup.



### Verifikasi data di PostgreSQL

In [22]:
import psycopg2
import os

conn = psycopg2.connect(os.getenv('DATABASE_URL'))

tables = ['dim_waktu', 'dim_negara', 'dim_komoditas', 'dim_indikator', 'fact_harga_pangan']
with conn.cursor() as cur:
    for t in tables:
        cur.execute(f'SELECT COUNT(*) FROM {t}')
        count = cur.fetchone()[0]
        print(f'{t:30s}: {count:,} baris')

conn.close()

dim_waktu                     : 24 baris
dim_negara                    : 6 baris
dim_komoditas                 : 152 baris
dim_indikator                 : 4 baris
fact_harga_pangan             : 3,834 baris
